In [ ]:
# check gpu
!nvidia-smi
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), torch.cuda.is_bf16_supported())

In [ ]:
# mount drive
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/TRM-BSS"
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# clone repo
%cd /content
!test -d TRM-BSS || git clone https://github.com/zi-wa/TRM-BSS.git
%cd /content/TRM-BSS
!git pull
!git log --oneline -1

In [ ]:
# install requirements
!grep -vxE "(torch|triton|adam-atan2) *" requirements.txt > /content/requirements_colab.txt
!pip install -q -r /content/requirements_colab.txt pytest
!pip install --no-cache-dir --no-build-isolation adam-atan2==0.0.3

In [ ]:
# check adam-atan2
from pretrain import AdamATan2
param = torch.nn.Parameter(torch.randn(8, device="cuda"))
optimizer = AdamATan2([param], lr=1e-3, weight_decay=0.1, betas=(0.9, 0.95))
param.square().sum().backward()
optimizer.step()
print("adam-atan2 ok")

In [ ]:
# unit tests
!python -m pytest -q tests

In [ ]:
# wandb offline
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_DIR"] = DRIVE_DIR

In [ ]:
# make tiny arc
import json
os.makedirs("data/tiny", exist_ok=True)
for subset, num_puzzles in [("training", 16), ("evaluation", 8)]:
    for kind in ["challenges", "solutions"]:
        with open(f"kaggle/combined/arc-agi_{subset}_{kind}.json") as f:
            puzzles = json.load(f)
        with open(f"data/tiny/arc-agi_{subset}_{kind}.json", "w") as f:
            json.dump({k: puzzles[k] for k in sorted(puzzles)[:num_puzzles]}, f)

In [ ]:
# build tiny dataset
!python -m dataset.build_arc_dataset --input-file-prefix data/tiny/arc-agi --output-dir data/arc-tiny --subsets training evaluation --test-set-name evaluation --num-aug 8

In [ ]:
# train trm
!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  arch=trm data_paths="[data/arc-tiny]" \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 \
  global_batch_size=32 epochs=200 eval_interval=50 lr_warmup_steps=50 ema=True \
  +run_name=tiny-trm +checkpoint_path={DRIVE_DIR}/checkpoints/tiny-trm

In [ ]:
# reload checkpoint
import glob, re
ckpt = max((p for p in glob.glob(f"{DRIVE_DIR}/checkpoints/tiny-trm/step_*") if re.search(r"step_\d+$", p)), key=lambda p: int(p.rsplit("_", 1)[1]))
print(ckpt)
!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  arch=trm data_paths="[data/arc-tiny]" \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 \
  global_batch_size=32 epochs=1 eval_interval=1 lr=0 puzzle_emb_lr=0 \
  +load_checkpoint={ckpt} +run_name=tiny-trm-reload +checkpoint_path={DRIVE_DIR}/checkpoints/tiny-trm-reload

In [ ]:
# train diffusion
!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  arch=trm_diffusion data_paths="[data/arc-tiny]" \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 \
  global_batch_size=32 epochs=200 eval_interval=50 lr_warmup_steps=50 ema=True \
  +run_name=tiny-diffusion +checkpoint_path={DRIVE_DIR}/checkpoints/tiny-diffusion

In [ ]:
# train masked diffusion
!torchrun --nproc-per-node 1 --rdzv_backend=c10d --rdzv_endpoint=localhost:0 --nnodes=1 pretrain.py \
  arch=trm_masked_diffusion data_paths="[data/arc-tiny]" \
  arch.L_layers=2 arch.H_cycles=3 arch.L_cycles=4 \
  global_batch_size=32 epochs=200 eval_interval=50 lr_warmup_steps=50 ema=True \
  +run_name=tiny-masked-diffusion +checkpoint_path={DRIVE_DIR}/checkpoints/tiny-masked-diffusion